In [1]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")
import pandas as pd
from src.messages import score_message
from src.transactions import score_transaction
from src.risk import decide
from src.explain import redact, explain

In [2]:
tricky = pd.DataFrame([
    # ---- should be ALLOWED (normal life that might look scary) ----
    ["k1",  "rosa",   "Grandma, I'm at the store. Do you want me to pick up milk and bread for you?", 48.20, "Kroger", "grocery", 0, "allow", "real family text + normal grocery run"],
    ["k2",  "walter", "Hi Dad, the pharmacy called. Your refill is ready whenever you want to grab it.", 36.50, "Walgreens", "pharmacy", 0, "allow", "family text that sounds urgent, normal payment"],
    ["k3",  "linda",  "Your new TV was delivered today. Thanks for shopping with us!", 649.99, "Best Buy", "electronics", 1, "allow", "big purchase at a new store, no scam signs"],
    ["k8",  "rosa",   "Your water bill is ready to view online.", 41.75, "City Water", "utilities", 1, "allow", "new payee, but a normal bill"],
    ["k9",  "linda",  "Security alert: we noticed an unusual login. Confirm your identity now or your card will be frozen.", 23.40, "Target", "retail", 0, "allow", "scam text, but the payment itself is normal"],
    # ---- should be HELD (scams that look different from Rosa's) ----
    ["k4",  "rosa",   "Grandma please don't tell Mom, I crashed the car and need $2,000 for the lawyer today. Send it through Western Union.", 2000.00, "Western Union", "wire", 1, "hold_and_alert", "grandparent scam, wire instead of gift cards"],
    ["k5",  "walter", "Microsoft Support: your computer is infected and your bank account is at risk. Buy Google Play cards to fix it.", 300.00, "Google Play", "gift_card", 1, "hold_and_alert", "tech-support scam"],
    ["k6",  "linda",  "Congratulations! You won a $5,000 prize. Pay the $50 processing fee with gift cards to claim it.", 50.00, "Target Gift Cards", "gift_card", 1, "hold_and_alert", "prize scam with a SMALL amount"],
    ["k7",  "walter", "See you at church on Sunday!", 1800.00, "Crypto ATM", "wire", 1, "hold_and_alert", "no scam text, but a very unusual risky payment"],
    ["k10", "walter", "My love, I need $900 for my visa so I can finally come see you. Please send it today, it's urgent.", 900.00, "Zelle to M. Brooks", "wire", 1, "hold_and_alert", "romance scam"],
], columns=["case_id", "user_id", "text", "amount", "merchant", "merchant_category", "is_new_payee", "expected_action", "why"])

tricky["timestamp"] = "2026-10-17 15:00"
tricky.to_csv("/home/ibm-user/shared/safecircle/scenarios/tricky.csv", index=False)
print(f"{len(tricky)} tricky cases saved")
tricky[["case_id", "user_id", "expected_action", "why"]]

10 tricky cases saved


,case_id,user_id,expected_action,why
0,k1,rosa,allow,real family text + normal grocery run
1,k2,walter,allow,"family text that sounds urgent, normal payment"
2,k3,linda,allow,"big purchase at a new store, no scam signs"
3,k8,rosa,allow,"new payee, but a normal bill"
4,k9,linda,allow,"scam text, but the payment itself is normal"
5,k4,rosa,hold_and_alert,"grandparent scam, wire instead of gift cards"
6,k5,walter,hold_and_alert,tech-support scam
7,k6,linda,hold_and_alert,prize scam with a SMALL amount
8,k7,walter,hold_and_alert,"no scam text, but a very unusual risky payment"
9,k10,walter,hold_and_alert,romance scam


In [3]:
rows = []
for _, c in tricky.iterrows():
    txn = {"user_id": c["user_id"], "txn_id": c["case_id"], "timestamp": c["timestamp"],
           "amount": c["amount"], "merchant": c["merchant"],
           "merchant_category": c["merchant_category"], "is_new_payee": c["is_new_payee"]}
    msg_r = score_message(c["text"])
    txn_r = score_transaction(txn)
    dec   = decide(msg_r, txn_r)
    rows.append({"case": c["case_id"], "expected": c["expected_action"], "got": dec["action"],
                 "PASS": dec["action"] == c["expected_action"],
                 "msg": round(msg_r["score"], 2), "txn": round(txn_r["score"], 2), "risk": round(dec["risk"], 2),
                 "why": c["why"]})

results = pd.DataFrame(rows)
print(f"Tricky set: {results['PASS'].sum()} / {len(results)} correct")
results

Tricky set: 7 / 10 correct


,case,expected,got,PASS,msg,txn,risk,why
0,k1,allow,allow,True,0.17,0.00,0.09,real family text + normal grocery run
1,k2,allow,allow,True,0.20,0.00,0.10,"family text that sounds urgent, normal payment"
2,k3,allow,hold_and_alert,False,0.63,1.00,0.81,"big purchase at a new store, no scam signs"
3,k8,allow,allow,True,0.31,0.37,0.34,"new payee, but a normal bill"
4,k9,allow,allow,True,0.73,0.14,0.44,"scam text, but the payment itself is normal"
5,k4,hold_and_alert,hold_and_alert,True,0.55,1.00,0.78,"grandparent scam, wire instead of gift cards"
6,k5,hold_and_alert,hold_and_alert,True,0.59,1.00,0.79,tech-support scam
7,k6,hold_and_alert,hold_and_alert,True,0.92,0.50,0.71,prize scam with a SMALL amount
8,k7,hold_and_alert,allow,False,0.10,1.00,0.55,"no scam text, but a very unusual risky payment"
9,k10,hold_and_alert,allow,False,0.09,1.00,0.54,romance scam


In [3]:
# Reusable runner: same logic as Cell 3, wrapped in a function so we can run any set of cases
def run_cases(cases, label):
    rows = []
    for _, c in cases.iterrows():
        txn = {"user_id": c["user_id"], "txn_id": c["case_id"], "timestamp": c["timestamp"],
               "amount": c["amount"], "merchant": c["merchant"],
               "merchant_category": c["merchant_category"], "is_new_payee": c["is_new_payee"]}
        msg_r = score_message(c["text"])
        txn_r = score_transaction(txn)
        dec   = decide(msg_r, txn_r)
        rows.append({"case": c["case_id"], "expected": c["expected_action"], "got": dec["action"],
                     "PASS": dec["action"] == c["expected_action"],
                     "msg": round(msg_r["score"], 2), "txn": round(txn_r["score"], 2),
                     "risk": round(dec["risk"], 2), "why": c["why"]})
    out = pd.DataFrame(rows)
    print(f"{label}: {out['PASS'].sum()} / {len(out)} correct")
    return out

# HOLDOUT: written after seeing k1-k10 results, never used for tuning. Run ONCE, after all fixes.
holdout = pd.DataFrame([
    ["k11", "linda",  "Your Uber is arriving in 3 minutes. Look for a gray Toyota.", 14.25, "Uber", "transport", 0, "allow", "normal ride, normal charge"],
    ["k12", "rosa",   "Hi Grandma! Thanks for the birthday card, I loved it. Talk soon!", 62.10, "Walgreens", "pharmacy", 0, "allow", "real grandchild text, normal purchase"],
    ["k13", "walter", "This is the Sheriff's office. You missed jury duty and must pay a $400 fine today with gift cards to avoid arrest.", 400.00, "Best Buy Gift Cards", "gift_card", 1, "hold_and_alert", "government-impersonation scam"],
    ["k14", "linda",  "Hey it's Pastor Mike, I need a favor. Can you buy four $100 Apple gift cards for a church member in need? I'll pay you back Sunday.", 400.00, "Apple Gift Cards", "gift_card", 1, "hold_and_alert", "trusted-person impersonation scam"],
], columns=["case_id", "user_id", "text", "amount", "merchant", "merchant_category", "is_new_payee", "expected_action", "why"])
holdout["timestamp"] = "2026-10-17 15:00"
holdout.to_csv("/home/ibm-user/shared/safecircle/scenarios/holdout.csv", index=False)
print(f"{len(holdout)} holdout cases saved. Do NOT run them until all fixes are in.")

4 holdout cases saved. Do NOT run them until all fixes are in.


In [4]:
after_gabe = run_cases(tricky, "Tricky set after Gabe's fix")
after_gabe

Tricky set after Gabe's fix: 10 / 10 correct


,case,expected,got,PASS,msg,txn,risk,why
0,k1,allow,allow,True,0.17,0.00,0.09,real family text + normal grocery run
1,k2,allow,allow,True,0.20,0.00,0.10,"family text that sounds urgent, normal payment"
2,k3,allow,allow,True,0.63,1.00,0.59,"big purchase at a new store, no scam signs"
3,k8,allow,allow,True,0.31,0.37,0.34,"new payee, but a normal bill"
4,k9,allow,allow,True,0.73,0.14,0.44,"scam text, but the payment itself is normal"
5,k4,hold_and_alert,hold_and_alert,True,0.55,1.00,0.78,"grandparent scam, wire instead of gift cards"
6,k5,hold_and_alert,hold_and_alert,True,0.59,1.00,0.79,tech-support scam
7,k6,hold_and_alert,hold_and_alert,True,0.92,0.50,0.71,prize scam with a SMALL amount
8,k7,hold_and_alert,hold_and_alert,True,0.10,1.00,0.60,"no scam text, but a very unusual risky payment"
9,k10,hold_and_alert,hold_and_alert,True,0.09,1.00,0.60,romance scam


In [4]:
final = run_cases(tricky, "Tricky set FINAL")
final

Tricky set FINAL: 10 / 10 correct


,case,expected,got,PASS,msg,txn,risk,why
0,k1,allow,allow,True,0.21,0.00,0.11,real family text + normal grocery run
1,k2,allow,allow,True,0.20,0.00,0.10,"family text that sounds urgent, normal payment"
2,k3,allow,allow,True,0.54,1.00,0.49,"big purchase at a new store, no scam signs"
3,k8,allow,allow,True,0.31,0.37,0.34,"new payee, but a normal bill"
4,k9,allow,allow,True,0.76,0.14,0.45,"scam text, but the payment itself is normal"
5,k4,hold_and_alert,hold_and_alert,True,0.68,1.00,0.85,"grandparent scam, wire instead of gift cards"
6,k5,hold_and_alert,hold_and_alert,True,0.63,1.00,0.85,tech-support scam
7,k6,hold_and_alert,hold_and_alert,True,0.95,0.50,0.85,prize scam with a SMALL amount
8,k7,hold_and_alert,hold_and_alert,True,0.11,1.00,0.85,"no scam text, but a very unusual risky payment"
9,k10,hold_and_alert,hold_and_alert,True,0.22,1.00,0.85,romance scam


In [5]:
fresh = run_cases(holdout, "Holdout (never tuned on)")
fresh

Holdout (never tuned on): 4 / 4 correct


,case,expected,got,PASS,msg,txn,risk,why
0,k11,allow,allow,True,0.39,0.22,0.30,"normal ride, normal charge"
1,k12,allow,allow,True,0.21,0.00,0.10,"real grandchild text, normal purchase"
2,k13,hold_and_alert,hold_and_alert,True,0.67,1.00,0.85,government-impersonation scam
3,k14,hold_and_alert,hold_and_alert,True,0.30,0.92,0.85,trusted-person impersonation scam
